## Autodesk APAC Inside Sales — RL-only (v32 final with all recommendations)

**Improvements in this version:**
1. Reward scaling (÷10 000)
2. Time‑based train/test split
3. Increased exploration (eps=0.3)
4. Gap embeddings + discretisation
5. **Double Q‑learning with target networks**
6. **Candidate action mask as state feature**
7. **Normalised tabular features** (StandardScaler)
8. **Handoff feature** (`days_since_handoff`)
9. **GRU encoder** (smaller, faster, less overfitting)
10. **Capacity‑aware target policy**
11. **Wait penalty** (discourages repeated waiting)
12. Hyperparameter tuning recommendations

In [1]:
# pip install numpy pandas scikit-learn torch matplotlib
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import defaultdict, deque
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", device)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

Torch device: cuda


In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)



## 1) Define actions + utilities

In [3]:
ACTION_TYPES = ["call", "email", "linkedin_msg", "demo_schedule", "nurture", "wait"]
K = len(ACTION_TYPES)
a_to_idx = {a:i for i,a in enumerate(ACTION_TYPES)}
idx_to_a = {i:a for a,i in a_to_idx.items()}

ACTION_COST = {
    "email": 1.0,
    "linkedin_msg": 1.0,
    "nurture": 1.0,
    "call": 2.0,
    "demo_schedule": 2.0,
    "wait": 0.0,
}

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max()
    e = np.exp(z)
    return e / (e.sum() + 1e-12)

## 2) Synthetic generator (APAC, inside sales, handoffs, action coverage)

**Updates:** `eps_explore = 0.3`, reward scaling (÷10000), and we add `days_since_handoff` to the generated snap.

In [4]:
APAC_COUNTRIES = ["IN","SG","AU","JP","KR","ID","MY","TH","PH","VN"]
SOURCES = ["webinar","website","linkedin","event","partner"]
PERSONAS = ["Student","Individual","Manager","Engineer","SMB_Owner"]
INDUSTRIES = ["AEC","Manufacturing","Retail","Energy","Media_Ent","Education","Auto","Consumer"]

SEGMENTS = ["midmarket","territory"]

@dataclass
class GenConfig:
    n_leads: int = 600
    n_days: int = 35
    start_date: str = "2026-01-01"
    handoff_rate: float = 0.15
    hist_n: int = 10
    eps_explore: float = 0.30          # increased from 0.15
    gamma: float = 0.98

cfg = GenConfig()

In [5]:
def make_ids(prefix, n, width=6):
    return [f"{prefix}{i:0{width}d}" for i in range(1, n+1)]

def gen_reps(countries):
    rows = []
    rid = 1
    for c in countries:
        n_terr = 5 if c in ["IN","JP","AU"] else 3
        n_mm = 3 if c in ["IN","JP","AU"] else 2
        for seg, n in [("territory", n_terr), ("midmarket", n_mm)]:
            for _ in range(n):
                rep_id = f"R{rid:04d}"
                rid += 1
                tenure = int(rng.integers(3, 48))
                skill = float(np.clip(rng.normal(0.0, 0.6), -1.5, 1.5))
                cap = int(rng.integers(35, 80)) if seg=="territory" else int(rng.integers(20, 60))
                rows.append({
                    "rep_id": rep_id,
                    "country_rep": c,
                    "segment_scope": seg,
                    "daily_capacity": cap,
                    "tenure_months": tenure,
                    "skill_score": skill
                })
    return pd.DataFrame(rows)

def gen_leads(n_leads, countries):
    lead_ids = make_ids("L", n_leads, width=7)
    country = rng.choice(countries, size=n_leads, p=np.array([0.22,0.08,0.10,0.12,0.10,0.10,0.08,0.07,0.07,0.06]))
    segment = rng.choice(["midmarket","territory"], size=n_leads, p=[0.40,0.60])
    source = rng.choice(SOURCES, size=n_leads, p=[0.28,0.28,0.18,0.18,0.08])
    persona = rng.choice(PERSONAS, size=n_leads, p=[0.20,0.20,0.20,0.20,0.20])
    industry = rng.choice(INDUSTRIES, size=n_leads)

    mu = np.where(segment=="midmarket", 0.58, 0.46)
    latent_q = np.clip(rng.normal(mu, 0.18), 0, 1)

    company_size = np.where(segment=="midmarket", rng.integers(200, 2000, size=n_leads), np.nan)
    miss = (segment=="territory") & (rng.random(n_leads) < 0.65)
    company_size[miss] = np.nan

    created = pd.to_datetime(cfg.start_date) + pd.to_timedelta(rng.integers(0, 20, size=n_leads), unit="D")

    revenue_band = np.where(segment=="midmarket",
                            rng.choice(["MM_Low","MM_Mid"], size=n_leads, p=[0.6,0.4]),
                            rng.choice(["Student","SMB"], size=n_leads, p=[0.35,0.65]))
    is_existing = rng.choice([0,1], size=n_leads, p=[0.78,0.22])

    return pd.DataFrame({
        "lead_id": lead_ids,
        "country": country,
        "segment": segment,
        "company_size": company_size,
        "persona": persona,
        "source": source,
        "created_date": pd.to_datetime(created).date,
        "industry": industry,
        "revenue_band": revenue_band,
        "is_existing_customer": is_existing,
        "latent_quality": latent_q
    })

def assign_owners(dim_lead, dim_rep, handoff_rate=0.15):
    rep_by_country_seg = defaultdict(list)
    for _, r in dim_rep.iterrows():
        rep_by_country_seg[(r["country_rep"], r["segment_scope"])].append(r["rep_id"])

    init_rep = []
    handoff_rep = []
    handoff_offset = []

    for _, row in dim_lead.iterrows():
        pool = rep_by_country_seg[(row["country"], row["segment"])]
        r0 = rng.choice(pool)
        init_rep.append(r0)
        if (len(pool) > 1) and (rng.random() < handoff_rate):
            r1 = rng.choice([x for x in pool if x != r0])
            handoff_rep.append(r1)
            handoff_offset.append(int(rng.integers(5, 26)))
        else:
            handoff_rep.append(None)
            handoff_offset.append(-1)

    dim_lead = dim_lead.copy()
    dim_lead["init_rep_id"] = init_rep
    dim_lead["handoff_rep_id"] = handoff_rep
    dim_lead["handoff_day_offset"] = handoff_offset
    return dim_lead

def owner_at_date(lead_row, asof_date):
    r = lead_row["init_rep_id"]
    off = int(lead_row["handoff_day_offset"])
    if off >= 0 and lead_row["handoff_rep_id"] is not None:
        hd = pd.to_datetime(lead_row["created_date"]) + pd.to_timedelta(off, unit="D")
        if pd.to_datetime(asof_date) >= hd:
            r = lead_row["handoff_rep_id"]
    return r

In [6]:
def generate_snap(dim_lead, dim_rep, cfg: GenConfig):
    lead_lookup = dim_lead.set_index("lead_id").to_dict(orient="index")
    rep_lookup = dim_rep.set_index("rep_id").to_dict(orient="index")

    start = pd.to_datetime(cfg.start_date).date()
    hist_n = cfg.hist_n

    lead_hist = defaultdict(lambda: deque([], maxlen=hist_n))
    rep_hist  = defaultdict(lambda: deque([], maxlen=hist_n))

    evt_dates = defaultdict(deque)
    tch_dates = defaultdict(deque)
    act_dates = defaultdict(deque)

    rep_load = defaultdict(int)

    snap_rows = []
    action_rows = []
    rec_id = 1

    for lead_id, L in lead_lookup.items():
        created = pd.to_datetime(L["created_date"]).date()
        latent_q = float(L["latent_quality"])
        seg = L["segment"]
        country = L["country"]
        src = str(L["source"])

        base_value = 8000 if seg=="territory" else 45000
        if not np.isnan(L["company_size"]):
            base_value *= (0.8 + 0.0004*float(L["company_size"]))
        opp_value = float(np.clip(rng.normal(base_value, 0.35*base_value), 1000, 250000))

        handoff_day = None
        if L["handoff_day_offset"] >= 0:
            handoff_day = created + pd.to_timedelta(int(L["handoff_day_offset"]), unit="D")

        for day in range(cfg.n_days):
            d = created + pd.to_timedelta(day, unit="D")
            d = pd.to_datetime(d).date()

            if d < start:
                continue

            rep_id = owner_at_date(L, d)
            R = rep_lookup[rep_id]

            # days since handoff (0 if no handoff or before handoff)
            if handoff_day is not None and d >= handoff_day:
                days_since_handoff = (pd.to_datetime(d) - pd.to_datetime(handoff_day)).days
            else:
                days_since_handoff = 0

            lam_evt = 0.15 + 1.3*latent_q + (0.3 if src in ["webinar","event"] else 0.0)
            lam_tch = 0.10 + 1.0*latent_q + (0.35 if src=="linkedin" else 0.0)

            n_evt = int(rng.poisson(lam_evt))
            n_tch = int(rng.poisson(lam_tch))

            if n_evt > 0:
                for _ in range(n_evt):
                    evt_dates[lead_id].append(d)
            if n_tch > 0:
                for _ in range(n_tch):
                    tch_dates[lead_id].append(d)

            def prune(q):
                while q and (pd.to_datetime(d) - pd.to_datetime(q[0])).days > 30:
                    q.popleft()

            prune(evt_dates[lead_id]); prune(tch_dates[lead_id]); prune(act_dates[lead_id])

            evt_cnt_30d = len(evt_dates[lead_id])
            tch_cnt_30d = len(tch_dates[lead_id])
            act_cnt_30d = len(act_dates[lead_id])

            evt_cnt_7d = sum((pd.to_datetime(d) - pd.to_datetime(x)).days <= 7 for x in evt_dates[lead_id])
            tch_cnt_7d = sum((pd.to_datetime(d) - pd.to_datetime(x)).days <= 7 for x in tch_dates[lead_id])
            act_cnt_7d = sum((pd.to_datetime(d) - pd.to_datetime(x)).days <= 7 for x in act_dates[lead_id])

            evt_intensity_sum_30d = float(evt_cnt_30d + 0.4*tch_cnt_30d)
            evt_intensity_mean_30d = float(evt_intensity_sum_30d / max(1, 30))

            # build histories
            lh = list(lead_hist[lead_id])[::-1]
            lead_tok = [0]*hist_n
            lead_gap = [np.nan]*hist_n
            for k, (aidx, adate) in enumerate(lh[:hist_n]):
                lead_tok[k] = int(aidx) + 1
                lead_gap[k] = float((pd.to_datetime(d) - pd.to_datetime(adate)).days)

            rh = list(rep_hist[(lead_id, rep_id)])[::-1]
            rep_tok = [0]*hist_n
            rep_gap = [np.nan]*hist_n
            for k, (aidx, adate) in enumerate(rh[:hist_n]):
                rep_tok[k] = int(aidx) + 1
                rep_gap[k] = float((pd.to_datetime(d) - pd.to_datetime(adate)).days)

            num_lead_actions = int(sum(t > 0 for t in lead_tok))
            num_rep_actions = int(sum(t > 0 for t in rep_tok))
            rep_cold_start = int(num_rep_actions == 0)

            days_since_prev_action = lead_gap[0]
            days_since_prev_rep_action = rep_gap[0]

            cand = ["email","nurture","wait"]
            phone_ok = (rng.random() < (0.55 if seg=="midmarket" else 0.35))
            if phone_ok:
                cand.append("call")
            if (src=="linkedin") or (tch_cnt_30d >= 4 and rng.random() < 0.4):
                cand.append("linkedin_msg")
            if seg=="midmarket" and latent_q > 0.55 and evt_cnt_30d >= 2:
                cand.append("demo_schedule")
            cand = list(dict.fromkeys(cand))

            # candidate action mask (binary vector)
            cand_mask = np.zeros(K, dtype=np.float32)
            for a in cand:
                cand_mask[a_to_idx[a]] = 1.0

            fatigue = 0.10*act_cnt_7d
            q = float(np.clip(0.55*latent_q + 0.25*sigmoid(0.25*(evt_cnt_30d + tch_cnt_30d)) + 0.15*sigmoid(R["skill_score"]) - fatigue, 0, 1))
            seg_boost = 0.25 if seg=="midmarket" else 0.0

            logits = {
                "email": 0.6 + 0.9*q + 0.10*R["skill_score"],
                "call": -0.1 + 1.6*q + seg_boost + 0.15*R["skill_score"],
                "nurture": 0.3 + 1.3*(1.0-q) + 0.05*R["skill_score"],
                "demo_schedule": -0.6 + 2.0*q + 0.35*seg_boost + 0.10*R["skill_score"],
                "linkedin_msg": -0.4 + 1.2*q + (0.35 if src=="linkedin" else 0.0) + 0.05*R["skill_score"],
                "wait": -2.0,
            }

            z = np.array([logits[a] for a in cand], dtype=np.float64)
            p = softmax(z)
            p = (1.0 - cfg.eps_explore)*p + (cfg.eps_explore / len(cand))

            load_key = (rep_id, d)
            remaining = max(0, int(R["daily_capacity"]) - rep_load[load_key])

            act = rng.choice(cand, p=p)
            pb = float(p[cand.index(act)])

            cost = ACTION_COST[act]
            if remaining - cost < 0:
                act = "wait"
                pb = 1.0
                cost = 0.0

            rep_load[load_key] += int(cost)

            # reward simulation (scaled later)
            eng = float(sigmoid(0.20*(evt_cnt_30d + 0.6*tch_cnt_30d)))
            base = {
                "email": -0.3,
                "call": -0.2,
                "linkedin_msg": -0.45,
                "demo_schedule": -0.6,
                "nurture": -0.15,
                "wait": -2.0
            }[act]
            uplift = {
                "email": 1.2*q + 0.6*eng,
                "call": 1.5*q + 0.4*eng + 0.2*seg_boost,
                "linkedin_msg": 1.0*q + 0.7*eng,
                "demo_schedule": 1.8*q + 0.5*eng + 0.3*seg_boost,
                "nurture": 0.6*(1-q) + 0.3*eng,
                "wait": 0.0
            }[act]
            p_succ = float(sigmoid(base + uplift + 0.15*R["skill_score"]))
            success = rng.random() < p_succ

            value_gain = (0.18*opp_value if success else 0.0)
            if act == "wait" and q > 0.6:
                value_gain -= 0.02*opp_value

            LAMBDA_COST = 0.06
            raw_reward = float(value_gain - LAMBDA_COST*cost*opp_value/50000.0)
            # Wait penalty: discourage consecutive waits
            if act == "wait" and lead_tok[0] == a_to_idx["wait"]+1:  # previous action was wait
                raw_reward -= 0.01 * opp_value / 50000.0   # small penalty
            # Scale reward by dividing by 10000
            reward = raw_reward / 10000.0

            meeting_booked_14d = int(success and act in ["call","email","linkedin_msg"])
            demo_completed_30d = int(success and act == "demo_schedule")

            if act != "wait":
                aidx = a_to_idx[act]
                lead_hist[lead_id].append((aidx, d))
                rep_hist[(lead_id, rep_id)].append((aidx, d))
                act_dates[lead_id].append(d)
                prune(act_dates[lead_id])

                action_rows.append({
                    "lead_id": lead_id,
                    "rep_id": rep_id,
                    "action_date": d,
                    "action_type": act,
                    "effort_cost": cost,
                    "reward": reward,
                })

            cand_str = ",".join(cand)

            snap_rows.append({
                "recommendation_id": f"REC{rec_id:010d}",
                "recommendation_date": d,
                "rep_id": rep_id,
                "lead_id": lead_id,
                "state_features": np.nan,
                "candidate_actions": cand_str,
                "chosen_action": act,
                "behavior_prob": pb,
                "reward": reward,
                "country": country,
                "segment": seg,
                "company_size": L["company_size"],
                "persona": L["persona"],
                "source": src,
                "created_date": created,
                "init_rep_id": L["init_rep_id"],
                "handoff_rep_id": L["handoff_rep_id"],
                "handoff_day_offset": int(L["handoff_day_offset"]),
                "industry": L["industry"],
                "revenue_band": L["revenue_band"],
                "is_existing_customer": int(L["is_existing_customer"]),
                "country_rep": R["country_rep"],
                "segment_scope": R["segment_scope"],
                "daily_capacity": int(R["daily_capacity"]),
                "tenure_months": int(R["tenure_months"]),
                "skill_score": float(R["skill_score"]),
                "evt_cnt_7d": int(evt_cnt_7d),
                "evt_cnt_30d": int(evt_cnt_30d),
                "tch_cnt_7d": int(tch_cnt_7d),
                "tch_cnt_30d": int(tch_cnt_30d),
                "act_cnt_7d": int(act_cnt_7d),
                "act_cnt_30d": int(act_cnt_30d),
                **{f"hist_tok_{i+1}": int(lead_tok[i]) for i in range(hist_n)},
                **{f"hist_gap_{i+1}": lead_gap[i] for i in range(hist_n)},
                **{f"rep_hist_tok_{i+1}": int(rep_tok[i]) for i in range(hist_n)},
                **{f"rep_hist_gap_{i+1}": rep_gap[i] for i in range(hist_n)},
                "days_since_prev_action": days_since_prev_action,
                "days_since_prev_rep_action": days_since_prev_rep_action,
                "num_lead_actions_so_far": num_lead_actions,
                "num_rep_actions_so_far": num_rep_actions,
                "rep_cold_start": rep_cold_start,
                "evt_intensity_sum_30d": evt_intensity_sum_30d,
                "evt_intensity_mean_30d": evt_intensity_mean_30d,
                "meeting_booked_14d": meeting_booked_14d,
                "demo_completed_30d": demo_completed_30d,
                "opportunity_value": opp_value,
                "chosen_cost": float(cost),
                "days_since_handoff": days_since_handoff,
                "cand_mask_0": cand_mask[0],
                "cand_mask_1": cand_mask[1],
                "cand_mask_2": cand_mask[2],
                "cand_mask_3": cand_mask[3],
                "cand_mask_4": cand_mask[4],
                "cand_mask_5": cand_mask[5],
            })
            rec_id += 1

    snap = pd.DataFrame(snap_rows)
    fact_sales_action = pd.DataFrame(action_rows)
    return snap, fact_sales_action

dim_rep = gen_reps(APAC_COUNTRIES)
dim_lead = gen_leads(cfg.n_leads, APAC_COUNTRIES)
dim_lead = assign_owners(dim_lead, dim_rep, cfg.handoff_rate)

snap, fact_sales_action = generate_snap(dim_lead, dim_rep, cfg)

print("dim_rep:", dim_rep.shape, "dim_lead:", dim_lead.shape)
print("snap:", snap.shape, "fact_sales_action:", fact_sales_action.shape)
snap.head(3)

dim_rep: (59, 6) dim_lead: (600, 14)
snap: (21000, 90) fact_sales_action: (19068, 6)


,recommendation_id,recommendation_date,rep_id,lead_id,state_features,candidate_actions,chosen_action,behavior_prob,reward,country,segment,company_size,persona,source,created_date,init_rep_id,handoff_rep_id,handoff_day_offset,industry,revenue_band,is_existing_customer,country_rep,segment_scope,daily_capacity,tenure_months,skill_score,evt_cnt_7d,evt_cnt_30d,tch_cnt_7d,tch_cnt_30d,act_cnt_7d,act_cnt_30d,hist_tok_1,hist_tok_2,hist_tok_3,hist_tok_4,hist_tok_5,hist_tok_6,hist_tok_7,hist_tok_8,hist_tok_9,hist_tok_10,hist_gap_1,hist_gap_2,hist_gap_3,hist_gap_4,hist_gap_5,hist_gap_6,hist_gap_7,hist_gap_8,hist_gap_9,hist_gap_10,rep_hist_tok_1,rep_hist_tok_2,rep_hist_tok_3,rep_hist_tok_4,rep_hist_tok_5,rep_hist_tok_6,rep_hist_tok_7,rep_hist_tok_8,rep_hist_tok_9,rep_hist_tok_10,rep_hist_gap_1,rep_hist_gap_2,rep_hist_gap_3,rep_hist_gap_4,rep_hist_gap_5,rep_hist_gap_6,rep_hist_gap_7,rep_hist_gap_8,rep_hist_gap_9,rep_hist_gap_10,days_since_prev_action,days_since_prev_rep_action,num_lead_actions_so_far,num_rep_actions_so_far,rep_cold_start,evt_intensity_sum_30d,evt_intensity_mean_30d,meeting_booked_14d,demo_completed_30d,opportunity_value,chosen_cost,days_since_handoff,cand_mask_0,cand_mask_1,cand_mask_2,cand_mask_3,cand_mask_4,cand_mask_5
0,REC0000000001,2026-01-14,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.417365,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1.8,0.060000,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
1,REC0000000002,2026-01-15,R0031,L0000001,NaN,"email,nurture,wait,call",email,0.327682,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,1,1,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1,1,0,2.2,0.073333,1,0,9510.427858,1.0,0,1.0,1.0,0.0,0.0,1.0,1.0
2,REC0000000003,2026-01-16,R0031,L0000001,NaN,"email,nurture,wait",email,0.395752,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,2,2,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2,2,0,2.2,0.073333,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0


In [7]:
snap.head(100)

,recommendation_id,recommendation_date,rep_id,lead_id,state_features,candidate_actions,chosen_action,behavior_prob,reward,country,segment,company_size,persona,source,created_date,init_rep_id,handoff_rep_id,handoff_day_offset,industry,revenue_band,is_existing_customer,country_rep,segment_scope,daily_capacity,tenure_months,skill_score,evt_cnt_7d,evt_cnt_30d,tch_cnt_7d,tch_cnt_30d,act_cnt_7d,act_cnt_30d,hist_tok_1,hist_tok_2,hist_tok_3,hist_tok_4,hist_tok_5,hist_tok_6,hist_tok_7,hist_tok_8,hist_tok_9,hist_tok_10,hist_gap_1,hist_gap_2,hist_gap_3,hist_gap_4,hist_gap_5,hist_gap_6,hist_gap_7,hist_gap_8,hist_gap_9,hist_gap_10,rep_hist_tok_1,rep_hist_tok_2,rep_hist_tok_3,rep_hist_tok_4,rep_hist_tok_5,rep_hist_tok_6,rep_hist_tok_7,rep_hist_tok_8,rep_hist_tok_9,rep_hist_tok_10,rep_hist_gap_1,rep_hist_gap_2,rep_hist_gap_3,rep_hist_gap_4,rep_hist_gap_5,rep_hist_gap_6,rep_hist_gap_7,rep_hist_gap_8,rep_hist_gap_9,rep_hist_gap_10,days_since_prev_action,days_since_prev_rep_action,num_lead_actions_so_far,num_rep_actions_so_far,rep_cold_start,evt_intensity_sum_30d,evt_intensity_mean_30d,meeting_booked_14d,demo_completed_30d,opportunity_value,chosen_cost,days_since_handoff,cand_mask_0,cand_mask_1,cand_mask_2,cand_mask_3,cand_mask_4,cand_mask_5
0,REC0000000001,2026-01-14,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.417365,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1.8,0.060000,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
1,REC0000000002,2026-01-15,R0031,L0000001,NaN,"email,nurture,wait,call",email,0.327682,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,1,1,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1,1,0,2.2,0.073333,1,0,9510.427858,1.0,0,1.0,1.0,0.0,0.0,1.0,1.0
2,REC0000000003,2026-01-16,R0031,L0000001,NaN,"email,nurture,wait",email,0.395752,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,2,2,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2,2,0,2.2,0.073333,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
3,REC0000000004,2026-01-17,R0031,L0000001,NaN,"email,nurture,wait",email,0.367420,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,3,3,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,3,3,0,3.0,0.100000,1,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
4,REC0000000005,2026-01-18,R0031,L0000001,NaN,"email,nurture,wait,linkedin_msg",linkedin_msg,0.155303,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,4,4,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,4,4,0,3.0,0.100000,0,0,9510.427858,1.0,0,0.0,1.0,1.0,0.0,1.0,1.0
5,REC0000000006,2026-01-19,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.579945,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,3,3,5,5,5,5,3,2,2,2,5,0,0,0,0,0,1.0,2.0,3.0,4.0,5.0,NaN,NaN,NaN,NaN,NaN,3,2,2,2,5,0,0,0,0,0,1.0,2.0,3.0,4.0,5.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,5,5,0,5.0,0.166667,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
6,REC0000000007,2026-01-20,R0031,L0000001,NaN,"email,nurture,wait",wait,0.113881,0.000000,KR,territory,NaN,Manager,website,2026-01-14,R0031,None,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,4,4,5,5,6,6,5,3,2,2,2,5,0,0,0,0,1.0,2.0,3.0,4.0,5.0,6.0,NaN,NaN,NaN,NaN,5,3,2,2,2,5,0,0,0,0,1.0,2.0,3.0,4.0,5.0,6.0,NaN,NaN,NaN,NaN,1.0,1.0,6,6,0,6.0,0.200000

In [8]:
snap.to_csv("data.csv",header=True)

In [9]:
snap = pd.read_csv("data.csv")

In [10]:
print("Chosen action distribution:")
display(snap["chosen_action"].value_counts(normalize=True).round(3))
print("\nBehavior prob summary:")
display(snap["behavior_prob"].describe())
print("\nReward summary (scaled):")
display(snap["reward"].describe())

Chosen action distribution:


chosen_action
nurture          0.494
email            0.253
wait             0.092
call             0.072
linkedin_msg     0.059
demo_schedule    0.030
Name: proportion, dtype: float64


Behavior prob summary:


count    21000.000000
mean         0.350907
std          0.171341
min          0.057837
25%          0.205230
50%          0.347681
75%          0.514475
max          0.607954
Name: behavior_prob, dtype: float64


Reward summary (scaled):


count    2.100000e+04
mean     2.950601e-01
std      4.526976e-01
min     -1.795874e-01
25%     -9.610610e-07
50%      1.137866e-01
75%      2.399328e-01
max      2.234744e+00
Name: reward, dtype: float64

## 3) Preprocessing with normalisation and candidate mask

We now include `cand_mask_*` columns as part of the state, and normalise numeric features.

In [11]:
def add_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["recommendation_date"] = pd.to_datetime(df["recommendation_date"])
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["lead_age_days"] = (df["recommendation_date"] - df["created_date"]).dt.days.astype(float)
    df["dow"] = df["recommendation_date"].dt.dayofweek.astype(int)
    df = df.drop(columns=["recommendation_date","created_date"], errors="ignore")
    return df

snap_fe = add_date_features(snap)

EXCLUDE = {
    "recommendation_id", "lead_id", "rep_id",
    "state_features", "chosen_action", "reward", "behavior_prob",
    "candidate_actions", "chosen_cost",
    "opportunity_value",
    "init_rep_id", "handoff_rep_id"
}

SEQ_COLS = [c for c in snap_fe.columns if c.startswith("hist_tok_") or c.startswith("hist_gap_")
            or c.startswith("rep_hist_tok_") or c.startswith("rep_hist_gap_")]

# Keep cand_mask columns in state
MASK_COLS = ["cand_mask_0", "cand_mask_1", "cand_mask_2", "cand_mask_3", "cand_mask_4", "cand_mask_5"]

state_df = snap_fe.drop(columns=list(EXCLUDE) + SEQ_COLS, errors="ignore")

num_cols = state_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in state_df.columns if c not in num_cols]

print("State columns:", len(state_df.columns), "| numeric:", len(num_cols), "| categorical:", len(cat_cols))

State columns: 39 | numeric: 31 | categorical: 8


In [12]:
print (state_df.shape)
state_df.head()

(21000, 39)


,Unnamed: 0,country,segment,company_size,persona,source,handoff_day_offset,industry,revenue_band,is_existing_customer,country_rep,segment_scope,daily_capacity,tenure_months,skill_score,evt_cnt_7d,evt_cnt_30d,tch_cnt_7d,tch_cnt_30d,act_cnt_7d,act_cnt_30d,days_since_prev_action,days_since_prev_rep_action,num_lead_actions_so_far,num_rep_actions_so_far,rep_cold_start,evt_intensity_sum_30d,evt_intensity_mean_30d,meeting_booked_14d,demo_completed_30d,days_since_handoff,cand_mask_0,cand_mask_1,cand_mask_2,cand_mask_3,cand_mask_4,cand_mask_5,lead_age_days,dow
0,0,KR,territory,NaN,Manager,website,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,2,2,0,0,NaN,NaN,0,0,1,1.8,0.060000,0,0,0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,2
1,1,KR,territory,NaN,Manager,website,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,1,1,1.0,1.0,1,1,0,2.2,0.073333,1,0,0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,3
2,2,KR,territory,NaN,Manager,website,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,2,2,1.0,1.0,2,2,0,2.2,0.073333,0,0,0,0.0,1.0,0.0,0.0,1.0,1.0,2.0,4
3,3,KR,territory,NaN,Manager,website,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,3,3,1.0,1.0,3,3,0,3.0,0.100000,1,0,0,0.0,1.0,0.0,0.0,1.0,1.0,3.0,5
4,4,KR,territory,NaN,Manager,website,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,4,4,1.0,1.0,4,4,0,3.0,0.100000,0,0,0,0.0,1.0,1.0,0.0,1.0,1.0,4.0,6


In [13]:
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# Pipeline: impute, then scale numeric, one‑hot encode categorical
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", make_onehot())]), cat_cols)
], sparse_threshold=0.0)

X_tab = prep.fit_transform(state_df).astype(np.float32)
print("X_tab shape:", X_tab.shape)

X_tab shape: (21000, 77)


In [14]:
num_cols + cat_cols

['Unnamed: 0',
 'company_size',
 'handoff_day_offset',
 'is_existing_customer',
 'daily_capacity',
 'tenure_months',
 'skill_score',
 'evt_cnt_7d',
 'evt_cnt_30d',
 'tch_cnt_7d',
 'tch_cnt_30d',
 'act_cnt_7d',
 'act_cnt_30d',
 'days_since_prev_action',
 'days_since_prev_rep_action',
 'num_lead_actions_so_far',
 'num_rep_actions_so_far',
 'rep_cold_start',
 'evt_intensity_sum_30d',
 'evt_intensity_mean_30d',
 'meeting_booked_14d',
 'demo_completed_30d',
 'days_since_handoff',
 'cand_mask_0',
 'cand_mask_1',
 'cand_mask_2',
 'cand_mask_3',
 'cand_mask_4',
 'cand_mask_5',
 'lead_age_days',
 'dow',
 'country',
 'segment',
 'persona',
 'source',
 'industry',
 'revenue_band',
 'country_rep',
 'segment_scope']

## 4) Time-based train/test split

In [15]:

snap_temp = snap.copy()
snap_temp["rec_date"] = pd.to_datetime(snap_temp["recommendation_date"])
date_order = snap_temp["rec_date"].sort_values().unique()
split_day_idx = int(0.75 * len(date_order))
train_dates = date_order[:split_day_idx]
test_dates = date_order[split_day_idx:]

train_mask = snap_temp["rec_date"].isin(train_dates)
test_mask = snap_temp["rec_date"].isin(test_dates)

tr_idx = np.where(train_mask)[0]
te_idx = np.where(test_mask)[0]

print(f"Train size: {len(tr_idx)}, Test size: {len(te_idx)}")
print(f"Train date range: {train_dates.min()} to {train_dates.max()}")
print(f"Test date range: {test_dates.min()} to {test_dates.max()}")


Train size: 17754, Test size: 3246
Train date range: 2026-01-01 00:00:00 to 2026-02-09 00:00:00
Test date range: 2026-02-10 00:00:00 to 2026-02-23 00:00:00


In [16]:
snap_temp.head()

,Unnamed: 0,recommendation_id,recommendation_date,rep_id,lead_id,state_features,candidate_actions,chosen_action,behavior_prob,reward,country,segment,company_size,persona,source,created_date,init_rep_id,handoff_rep_id,handoff_day_offset,industry,revenue_band,is_existing_customer,country_rep,segment_scope,daily_capacity,tenure_months,skill_score,evt_cnt_7d,evt_cnt_30d,tch_cnt_7d,tch_cnt_30d,act_cnt_7d,act_cnt_30d,hist_tok_1,hist_tok_2,hist_tok_3,hist_tok_4,hist_tok_5,hist_tok_6,hist_tok_7,hist_tok_8,hist_tok_9,hist_tok_10,hist_gap_1,hist_gap_2,hist_gap_3,hist_gap_4,hist_gap_5,hist_gap_6,hist_gap_7,hist_gap_8,hist_gap_9,hist_gap_10,rep_hist_tok_1,rep_hist_tok_2,rep_hist_tok_3,rep_hist_tok_4,rep_hist_tok_5,rep_hist_tok_6,rep_hist_tok_7,rep_hist_tok_8,rep_hist_tok_9,rep_hist_tok_10,rep_hist_gap_1,rep_hist_gap_2,rep_hist_gap_3,rep_hist_gap_4,rep_hist_gap_5,rep_hist_gap_6,rep_hist_gap_7,rep_hist_gap_8,rep_hist_gap_9,rep_hist_gap_10,days_since_prev_action,days_since_prev_rep_action,num_lead_actions_so_far,num_rep_actions_so_far,rep_cold_start,evt_intensity_sum_30d,evt_intensity_mean_30d,meeting_booked_14d,demo_completed_30d,opportunity_value,chosen_cost,days_since_handoff,cand_mask_0,cand_mask_1,cand_mask_2,cand_mask_3,cand_mask_4,cand_mask_5,rec_date
0,0,REC0000000001,2026-01-14,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.417365,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1.8,0.060000,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0,2026-01-14
1,1,REC0000000002,2026-01-15,R0031,L0000001,NaN,"email,nurture,wait,call",email,0.327682,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,1,1,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1,1,0,2.2,0.073333,1,0,9510.427858,1.0,0,1.0,1.0,0.0,0.0,1.0,1.0,2026-01-15
2,2,REC0000000003,2026-01-16,R0031,L0000001,NaN,"email,nurture,wait",email,0.395752,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,2,2,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2,2,0,2.2,0.073333,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0,2026-01-16
3,3,REC0000000004,2026-01-17,R0031,L0000001,NaN,"email,nurture,wait",email,0.367420,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,3,3,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,3,3,0,3.0,0.100000,1,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0,2026-01-17
4,4,REC0000000005,2026-01-18,R0031,L0000001,NaN,"email,nurture,wait,linkedin_msg",linkedin_msg,0.155303,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,4,4,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,4,4,0,3.0,0.100000,0,0,9510.427858,1.0,0,0.0,1.0,1.0,0.0,1.0,1.0,2026-01-18


In [17]:
tr_idx

array([    0,     1,     2, ..., 20990, 20991, 20992], shape=(17754,))

## 5) Bandit reward model (baseline)

We still train a simple Ridge for comparison, but the main RL will use the Q‑network.

In [18]:
a_logged = snap["chosen_action"].map(a_to_idx).astype(int).values
r_logged = snap["reward"].astype(float).values
pb = snap["behavior_prob"].astype(float).values

A_oh = np.eye(K, dtype=np.float32)[a_logged]
X_sa = np.hstack([X_tab, A_oh]).astype(np.float32)

X_sa_tr, X_sa_te = X_sa[tr_idx], X_sa[te_idx]
r_tr, r_te = r_logged[tr_idx], r_logged[te_idx]

bandit = Ridge(alpha=1.0, random_state=SEED)
bandit.fit(X_sa_tr, r_tr)

def Q_hat_bandit(x_tab, a_idx):
    a_oh = np.eye(K, dtype=np.float32)[a_idx]
    x_sa = np.hstack([x_tab, a_oh]).astype(np.float32)
    return bandit.predict(x_sa)

Xte = X_tab[te_idx]
Q_bandit_te = np.stack([Q_hat_bandit(Xte, np.full(len(te_idx), a)) for a in range(K)], axis=1)

def parse_cand(s):
    return [a_to_idx[x] for x in str(s).split(",") if x in a_to_idx]

cand_list = [parse_cand(snap.loc[i, "candidate_actions"]) for i in te_idx]
mask = np.full((len(te_idx), K), False)
for i, cands in enumerate(cand_list):
    mask[i, cands] = True

Q_masked = Q_bandit_te.copy()
Q_masked[~mask] = -1e9
a_star_bandit = Q_masked.argmax(axis=1).astype(int)

print("Bandit target action distribution (test):")
print(pd.Series(a_star_bandit).map(idx_to_a).value_counts(normalize=True).round(3))

Bandit target action distribution (test):
nurture    1.0
Name: proportion, dtype: float64


In [19]:
Q_bandit_te

array([[-0.24426603, -0.25284255, -0.231206  , -0.5296419 ,  0.08318511,
        -0.18487343],
       [-0.23873642, -0.2473129 , -0.22567636, -0.5241123 ,  0.08871475,
        -0.17934379],
       [-0.23119944, -0.23977596, -0.21813941, -0.51657534,  0.0962517 ,
        -0.17180684],
       ...,
       [-0.2366597 , -0.24523619, -0.22359961, -0.52203554,  0.09079149,
        -0.17726704],
       [-0.23077163, -0.23934811, -0.21771157, -0.5161475 ,  0.09667954,
        -0.171379  ],
       [-0.22422484, -0.23280132, -0.21116477, -0.5096007 ,  0.10322633,
        -0.1648322 ]], shape=(3246, 6), dtype=float32)

In [20]:
Q_bandit_te

array([[-0.24426603, -0.25284255, -0.231206  , -0.5296419 ,  0.08318511,
        -0.18487343],
       [-0.23873642, -0.2473129 , -0.22567636, -0.5241123 ,  0.08871475,
        -0.17934379],
       [-0.23119944, -0.23977596, -0.21813941, -0.51657534,  0.0962517 ,
        -0.17180684],
       ...,
       [-0.2366597 , -0.24523619, -0.22359961, -0.52203554,  0.09079149,
        -0.17726704],
       [-0.23077163, -0.23934811, -0.21771157, -0.5161475 ,  0.09667954,
        -0.171379  ],
       [-0.22422484, -0.23280132, -0.21116477, -0.5096007 ,  0.10322633,
        -0.1648322 ]], shape=(3246, 6), dtype=float32)

In [21]:
Q_masked

array([[-1.00000000e+09, -2.52842546e-01, -2.31206000e-01,
        -1.00000000e+09,  8.31851065e-02, -1.84873432e-01],
       [-2.38736421e-01, -2.47312903e-01, -1.00000000e+09,
        -1.00000000e+09,  8.87147486e-02, -1.79343790e-01],
       [-1.00000000e+09, -2.39775956e-01, -1.00000000e+09,
        -1.00000000e+09,  9.62516963e-02, -1.71806842e-01],
       ...,
       [-1.00000000e+09, -2.45236188e-01, -2.23599613e-01,
        -1.00000000e+09,  9.07914937e-02, -1.77267045e-01],
       [-1.00000000e+09, -2.39348114e-01, -2.17711568e-01,
        -1.00000000e+09,  9.66795385e-02, -1.71379000e-01],
       [-1.00000000e+09, -2.32801318e-01, -1.00000000e+09,
        -1.00000000e+09,  1.03226334e-01, -1.64832205e-01]],
      shape=(3246, 6), dtype=float32)

In [22]:
a_star_bandit

array([4, 4, 4, ..., 4, 4, 4], shape=(3246,))

In [23]:
# Inverse Propensity Scoring

# What each input is
# a_logged: action actually taken in the dataset
# r_logged: reward actually observed for that logged action
# pb: probability that the behavior policy assigned to the logged action
# a_star: action chosen by the target policy you want to evaluate
# Q_model: model-predicted reward/value for every action in every row
# clip: caps large importance weights so estimates do not explode

In [24]:
def ope_ips_dr(a_logged, r_logged, pb, a_star, Q_model, clip=10.0):
    pi = (a_logged == a_star).astype(np.float32)
    w = pi / (pb + 1e-9)
    w = np.clip(w, 0, clip)
    ips = np.mean(w * r_logged)
    q_star = Q_model[np.arange(len(Q_model)), a_star]
    q_log  = Q_model[np.arange(len(Q_model)), a_logged]
    dr = np.mean(q_star + w*(r_logged - q_log))
    return ips, dr

def bootstrap_ci_ope(a_logged, r_logged, pb, a_star, Q_model, n_boot=300, clip=10.0, seed=SEED):
    rg = np.random.default_rng(seed)
    n = len(a_logged)
    ips_list, dr_list = [], []
    for _ in range(n_boot):
        b = rg.integers(0, n, size=n)
        ips, dr = ope_ips_dr(a_logged[b], r_logged[b], pb[b], a_star[b], Q_model[b], clip=clip)
        ips_list.append(ips); dr_list.append(dr)
    ips_arr = np.array(ips_list)
    dr_arr = np.array(dr_list)
    return {
        "ips_mean": float(ips_arr.mean()),
        "ips_lo": float(np.percentile(ips_arr, 5)),
        "ips_hi": float(np.percentile(ips_arr, 95)),
        "dr_mean": float(dr_arr.mean()),
        "dr_lo": float(np.percentile(dr_arr, 5)),
        "dr_hi": float(np.percentile(dr_arr, 95)),
    }

a_logged_te = a_logged[te_idx]
r_logged_te = r_logged[te_idx]
pb_te = pb[te_idx]

ips_bandit, dr_bandit = ope_ips_dr(a_logged_te, r_logged_te, pb_te, a_star_bandit, Q_bandit_te, clip=10.0)
print("OPE (Bandit policy) IPS:", round(float(ips_bandit),4), "DR:", round(float(dr_bandit),4))

ci_bandit = bootstrap_ci_ope(a_logged_te, r_logged_te, pb_te, a_star_bandit, Q_bandit_te, n_boot=300, clip=10.0)
ci_bandit

OPE (Bandit policy) IPS: 0.3338 DR: 0.4687


{'ips_mean': 0.334181693260685,
 'ips_lo': 0.31198956777317616,
 'ips_hi': 0.3566964513948884,
 'dr_mean': 0.46810212581231486,
 'dr_lo': 0.44624511781546755,
 'dr_hi': 0.48674505305889393}

## 6) Build RL transitions with gaps and candidate mask

## We now include `cand_mask` in the tabular state (already part of X_tab).

This block is converting the snap table into offline RL transitions of the form:
    (s,a,r,s′,done)

In [25]:
# lead_tok_cols

In [26]:
snap.head()

,Unnamed: 0,recommendation_id,recommendation_date,rep_id,lead_id,state_features,candidate_actions,chosen_action,behavior_prob,reward,country,segment,company_size,persona,source,created_date,init_rep_id,handoff_rep_id,handoff_day_offset,industry,revenue_band,is_existing_customer,country_rep,segment_scope,daily_capacity,tenure_months,skill_score,evt_cnt_7d,evt_cnt_30d,tch_cnt_7d,tch_cnt_30d,act_cnt_7d,act_cnt_30d,hist_tok_1,hist_tok_2,hist_tok_3,hist_tok_4,hist_tok_5,hist_tok_6,hist_tok_7,hist_tok_8,hist_tok_9,hist_tok_10,hist_gap_1,hist_gap_2,hist_gap_3,hist_gap_4,hist_gap_5,hist_gap_6,hist_gap_7,hist_gap_8,hist_gap_9,hist_gap_10,rep_hist_tok_1,rep_hist_tok_2,rep_hist_tok_3,rep_hist_tok_4,rep_hist_tok_5,rep_hist_tok_6,rep_hist_tok_7,rep_hist_tok_8,rep_hist_tok_9,rep_hist_tok_10,rep_hist_gap_1,rep_hist_gap_2,rep_hist_gap_3,rep_hist_gap_4,rep_hist_gap_5,rep_hist_gap_6,rep_hist_gap_7,rep_hist_gap_8,rep_hist_gap_9,rep_hist_gap_10,days_since_prev_action,days_since_prev_rep_action,num_lead_actions_so_far,num_rep_actions_so_far,rep_cold_start,evt_intensity_sum_30d,evt_intensity_mean_30d,meeting_booked_14d,demo_completed_30d,opportunity_value,chosen_cost,days_since_handoff,cand_mask_0,cand_mask_1,cand_mask_2,cand_mask_3,cand_mask_4,cand_mask_5
0,0,REC0000000001,2026-01-14,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.417365,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1.8,0.060000,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
1,1,REC0000000002,2026-01-15,R0031,L0000001,NaN,"email,nurture,wait,call",email,0.327682,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,1,1,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1,1,0,2.2,0.073333,1,0,9510.427858,1.0,0,1.0,1.0,0.0,0.0,1.0,1.0
2,2,REC0000000003,2026-01-16,R0031,L0000001,NaN,"email,nurture,wait",email,0.395752,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,2,2,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2,2,0,2.2,0.073333,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
3,3,REC0000000004,2026-01-17,R0031,L0000001,NaN,"email,nurture,wait",email,0.367420,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,3,3,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,3,3,0,3.0,0.100000,1,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
4,4,REC0000000005,2026-01-18,R0031,L0000001,NaN,"email,nurture,wait,linkedin_msg",linkedin_msg,0.155303,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,4,4,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,4,4,0,3.0,0.100000,0,0,9510.427858,1.0,0,0.0,1.0,1.0,0.0,1.0,1.0


In [27]:
# # Number of historical actions to keep in each sequence
# HIST_N = cfg.hist_n

# # Column names for the lead's recent action tokens, e.g. hist_tok_1 ... hist_tok_N
# lead_tok_cols = [f"hist_tok_{i}" for i in range(1, HIST_N+1)]

# # Column names for the lead's recent action time gaps, e.g. hist_gap_1 ... hist_gap_N
# lead_gap_cols = [f"hist_gap_{i}" for i in range(1, HIST_N+1)]

# # Column names for the rep-lead recent action tokens
# rep_tok_cols  = [f"rep_hist_tok_{i}" for i in range(1, HIST_N+1)]

# # Column names for the rep-lead recent action time gaps
# rep_gap_cols  = [f"rep_hist_gap_{i}" for i in range(1, HIST_N+1)]


# # Extract lead history action-token columns from snap
# # Missing values are replaced with 0, which is likely the padding / no-history token
# # Convert to int64 because token IDs are typically integer indices for embeddings
# lead_tok_all = snap[lead_tok_cols].fillna(0).astype(int).values.astype(np.int64)

# # Extract lead history gap columns from snap
# # Missing values are replaced with -1, likely used as a sentinel meaning "no valid gap"
# # Convert to float32 for neural network input
# lead_gap_all = snap[lead_gap_cols].fillna(-1).astype(np.float32).values

# # Extract rep-lead history action tokens
# rep_tok_all  = snap[rep_tok_cols].fillna(0).astype(int).values.astype(np.int64)

# # Extract rep-lead history gaps
# rep_gap_all  = snap[rep_gap_cols].fillna(-1).astype(np.float32).values


# # Make a copy so sorting does not alter the original snap DataFrame
# snap_sorted = snap.copy()

# # Ensure recommendation_date is actual datetime type for proper chronological sorting
# snap_sorted["recommendation_date"] = pd.to_datetime(snap_sorted["recommendation_date"])

# # Sort the rows by lead_id first, then by date within each lead
# # reset_index() keeps the original row index in a column named "index"
# snap_sorted = snap_sorted.sort_values(["lead_id","recommendation_date"]).reset_index()

# # Save the original row indices, now aligned to the sorted order
# # This lets us reorder previously created numpy arrays to match snap_sorted
# orig_idx = snap_sorted["index"].values


# # Create an array that will store the row index of the "next state" for each row
# # Start with -1, meaning "no next state"
# next_row = np.full(len(snap_sorted), -1, dtype=int)

# # Create terminal flags: 1.0 if this row is the last step in an episode, else 0.0
# done = np.zeros(len(snap_sorted), dtype=np.float32)

# # Group rows by lead_id, so each lead becomes one trajectory / episode
# for lid, inds in snap_sorted.groupby("lead_id").indices.items():
#     # Convert the row positions for this lead into a numpy array
#     inds = np.array(list(inds), dtype=int)

#     # If this lead has only one row, it is immediately terminal
#     if len(inds) == 1:
#         done[inds[0]] = 1.0
#     else:
#         # For every row except the last, point to the next row for the same lead
#         next_row[inds[:-1]] = inds[1:]

#         # Mark the last row for that lead as terminal
#         done[inds[-1]] = 1.0


# # Ensure tabular state features are float32
# X_tab_all = X_tab.astype(np.float32)

# # Reorder the tabular states into the same sorted order as snap_sorted
# S_tab = X_tab_all[orig_idx]

# # Reorder lead token histories into sorted trajectory order
# S_lead_tok = lead_tok_all[orig_idx]

# # Reorder lead gap histories into sorted trajectory order
# S_lead_gap = lead_gap_all[orig_idx]

# # Reorder rep token histories into sorted trajectory order
# S_rep_tok = rep_tok_all[orig_idx]

# # Reorder rep gap histories into sorted trajectory order
# S_rep_gap = rep_gap_all[orig_idx]

# # Logged action at each sorted row, converted from string label to integer action index
# A = snap_sorted["chosen_action"].map(a_to_idx).astype(int).values

# # Observed reward at each sorted row
# R = snap_sorted["reward"].astype(np.float32).values


# # Allocate arrays for next-state tabular features
# # Start with zeros; terminal rows will keep these zero placeholders
# S_next_tab = np.zeros_like(S_tab)

# # Allocate arrays for next-state lead token histories
# S_next_lead_tok = np.zeros_like(S_lead_tok)

# # Allocate arrays for next-state lead gap histories
# S_next_lead_gap = np.zeros_like(S_lead_gap)

# # Allocate arrays for next-state rep token histories
# S_next_rep_tok = np.zeros_like(S_rep_tok)

# # Allocate arrays for next-state rep gap histories
# S_next_rep_gap = np.zeros_like(S_rep_gap)

# # Boolean mask for rows that actually have a next state
# m = next_row != -1

# # For non-terminal rows, fill in next-state tabular features
# S_next_tab[m] = S_tab[next_row[m]]

# # For non-terminal rows, fill in next-state lead token history
# S_next_lead_tok[m] = S_lead_tok[next_row[m]]

# # For non-terminal rows, fill in next-state lead gap history
# S_next_lead_gap[m] = S_lead_gap[next_row[m]]

# # For non-terminal rows, fill in next-state rep token history
# S_next_rep_tok[m] = S_rep_tok[next_row[m]]

# # For non-terminal rows, fill in next-state rep gap history
# S_next_rep_gap[m] = S_rep_gap[next_row[m]]


# # Print total number of RL transitions and how many are terminal
# print("RL transitions:", len(S_tab), "terminal:", int(done.sum()))

In [46]:
snap.head(100)

,Unnamed: 0,recommendation_id,recommendation_date,rep_id,lead_id,state_features,candidate_actions,chosen_action,behavior_prob,reward,country,segment,company_size,persona,source,created_date,init_rep_id,handoff_rep_id,handoff_day_offset,industry,revenue_band,is_existing_customer,country_rep,segment_scope,daily_capacity,tenure_months,skill_score,evt_cnt_7d,evt_cnt_30d,tch_cnt_7d,tch_cnt_30d,act_cnt_7d,act_cnt_30d,hist_tok_1,hist_tok_2,hist_tok_3,hist_tok_4,hist_tok_5,hist_tok_6,hist_tok_7,hist_tok_8,hist_tok_9,hist_tok_10,hist_gap_1,hist_gap_2,hist_gap_3,hist_gap_4,hist_gap_5,hist_gap_6,hist_gap_7,hist_gap_8,hist_gap_9,hist_gap_10,rep_hist_tok_1,rep_hist_tok_2,rep_hist_tok_3,rep_hist_tok_4,rep_hist_tok_5,rep_hist_tok_6,rep_hist_tok_7,rep_hist_tok_8,rep_hist_tok_9,rep_hist_tok_10,rep_hist_gap_1,rep_hist_gap_2,rep_hist_gap_3,rep_hist_gap_4,rep_hist_gap_5,rep_hist_gap_6,rep_hist_gap_7,rep_hist_gap_8,rep_hist_gap_9,rep_hist_gap_10,days_since_prev_action,days_since_prev_rep_action,num_lead_actions_so_far,num_rep_actions_so_far,rep_cold_start,evt_intensity_sum_30d,evt_intensity_mean_30d,meeting_booked_14d,demo_completed_30d,opportunity_value,chosen_cost,days_since_handoff,cand_mask_0,cand_mask_1,cand_mask_2,cand_mask_3,cand_mask_4,cand_mask_5
0,0,REC0000000001,2026-01-14,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.417365,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1.8,0.060000,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
1,1,REC0000000002,2026-01-15,R0031,L0000001,NaN,"email,nurture,wait,call",email,0.327682,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,1,1,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0,0,0,0,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1,1,0,2.2,0.073333,1,0,9510.427858,1.0,0,1.0,1.0,0.0,0.0,1.0,1.0
2,2,REC0000000003,2026-01-16,R0031,L0000001,NaN,"email,nurture,wait",email,0.395752,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,3,3,2,2,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,5,0,0,0,0,0,0,0,0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2,2,0,2.2,0.073333,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
3,3,REC0000000004,2026-01-17,R0031,L0000001,NaN,"email,nurture,wait",email,0.367420,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,3,3,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,5,0,0,0,0,0,0,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,3,3,0,3.0,0.100000,1,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
4,4,REC0000000005,2026-01-18,R0031,L0000001,NaN,"email,nurture,wait,linkedin_msg",linkedin_msg,0.155303,-0.000001,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,1,1,5,5,4,4,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,5,0,0,0,0,0,0,1.0,2.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,4,4,0,3.0,0.100000,0,0,9510.427858,1.0,0,0.0,1.0,1.0,0.0,1.0,1.0
5,5,REC0000000006,2026-01-19,R0031,L0000001,NaN,"email,nurture,wait",nurture,0.579945,0.171187,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,3,3,5,5,5,5,3,2,2,2,5,0,0,0,0,0,1.0,2.0,3.0,4.0,5.0,NaN,NaN,NaN,NaN,NaN,3,2,2,2,5,0,0,0,0,0,1.0,2.0,3.0,4.0,5.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,5,5,0,5.0,0.166667,0,0,9510.427858,1.0,0,0.0,1.0,0.0,0.0,1.0,1.0
6,6,REC0000000007,2026-01-20,R0031,L0000001,NaN,"email,nurture,wait",wait,0.113881,0.000000,KR,territory,NaN,Manager,website,2026-01-14,R0031,NaN,-1,Retail,SMB,0,KR,territory,60,41,-0.200931,4,4,5,5,6,6,5,3,2,2,2,5,0,0,0,0,1.0,2.0,3.0,4.0,5.0,6.0,NaN,NaN,NaN,NaN,5,3,2,2,2,5,0,0,0,0,1.0,2.0,3.0,4.0,5.0,6.0,NaN,NaN,NaN,NaN,1.0,1.0,

In [53]:
i=9

snap[[f"hist_tok_{i}", f"hist_gap_{i}", f"rep_hist_tok_{i}", f"rep_hist_gap_{i}"]].head(1000)

,hist_tok_9,hist_gap_9,rep_hist_tok_9,rep_hist_gap_9
0,0,NaN,0,NaN
1,0,NaN,0,NaN
2,0,NaN,0,NaN
3,0,NaN,0,NaN
4,0,NaN,0,NaN
5,0,NaN,0,NaN
6,0,NaN,0,NaN
7,0,NaN,0,NaN
8,0,NaN,0,NaN
9,0,NaN,0,NaN


In [28]:
HIST_N = cfg.hist_n

lead_tok_cols = [f"hist_tok_{i}" for i in range(1, HIST_N+1)]
lead_gap_cols = [f"hist_gap_{i}" for i in range(1, HIST_N+1)]
rep_tok_cols  = [f"rep_hist_tok_{i}" for i in range(1, HIST_N+1)]
rep_gap_cols  = [f"rep_hist_gap_{i}" for i in range(1, HIST_N+1)]

lead_tok_all = snap[lead_tok_cols].fillna(0).astype(int).values.astype(np.int64)
lead_gap_all = snap[lead_gap_cols].fillna(-1).astype(np.float32).values
rep_tok_all  = snap[rep_tok_cols].fillna(0).astype(int).values.astype(np.int64)
rep_gap_all  = snap[rep_gap_cols].fillna(-1).astype(np.float32).values

# Sort by lead and date
snap_sorted = snap.copy()
snap_sorted["recommendation_date"] = pd.to_datetime(snap_sorted["recommendation_date"])
snap_sorted = snap_sorted.sort_values(["lead_id","recommendation_date"]).reset_index()
orig_idx = snap_sorted["index"].values

next_row = np.full(len(snap_sorted), -1, dtype=int)

done = np.zeros(len(snap_sorted), dtype=np.float32)
for lid, inds in snap_sorted.groupby("lead_id").indices.items():
    inds = np.array(list(inds), dtype=int)
    if len(inds) == 1:
        done[inds[0]] = 1.0
    else:
        next_row[inds[:-1]] = inds[1:]
        done[inds[-1]] = 1.0

X_tab_all = X_tab.astype(np.float32)
S_tab = X_tab_all[orig_idx]

S_lead_tok = lead_tok_all[orig_idx]
S_lead_gap = lead_gap_all[orig_idx]

S_rep_tok = rep_tok_all[orig_idx]
S_rep_gap = rep_gap_all[orig_idx]

A = snap_sorted["chosen_action"].map(a_to_idx).astype(int).values
R = snap_sorted["reward"].astype(np.float32).values

# Next state
S_next_tab = np.zeros_like(S_tab)

S_next_lead_tok = np.zeros_like(S_lead_tok)
S_next_lead_gap = np.zeros_like(S_lead_gap)

S_next_rep_tok = np.zeros_like(S_rep_tok)
S_next_rep_gap = np.zeros_like(S_rep_gap)

m = next_row != -1
S_next_tab[m] = S_tab[next_row[m]]

S_next_lead_tok[m] = S_lead_tok[next_row[m]]
S_next_lead_gap[m] = S_lead_gap[next_row[m]]

S_next_rep_tok[m] = S_rep_tok[next_row[m]]
S_next_rep_gap[m] = S_rep_gap[next_row[m]]

print("RL transitions:", len(S_tab), "terminal:", int(done.sum()))

RL transitions: 21000 terminal: 600


In [29]:
len(S_next_rep_gap)

21000

In [45]:
S_lead_tok == S_rep_tok

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ...,  True, False, False],
       [ True,  True,  True, ...,  True,  True, False]], shape=(21000, 10))

In [30]:
S_tab.shape

(21000, 77)

In [43]:
print (len(S_rep_tok))
print (len(S_next_rep_gap))

21000
21000


In [41]:
S_rep_gap[0]

array([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.], dtype=float32)

In [42]:
S_next_rep_gap[0]

array([ 1., -1., -1., -1., -1., -1., -1., -1., -1., -1.], dtype=float32)

## 7) GRU-based encoder with gap embeddings (smaller architecture)

We replace the Transformer with a two‑layer GRU for both lead and rep histories. This is more data‑efficient and faster.

Q-network → predicts value for each action

V-network → predicts value of the state only

In [31]:
K

6

In [33]:
tab_dim = S_tab.shape[1]

# ACTION_TYPES = ["call", "email", "linkedin_msg", "demo_schedule", "nurture", "wait"]
# K = len(ACTION_TYPES)

token_vocab = K + 1
gap_vocab = 8      # 0-5, 6=missing

def discretize_gap(gap, max_bin=5):
    if np.isnan(gap) or gap < 0:
        return 6
    return min(int(gap), max_bin)

class GRUEncoder(nn.Module):
    def __init__(self, token_vocab, gap_vocab, seq_len, d_model=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(token_vocab, d_model)
        self.gap_emb = nn.Embedding(gap_vocab, d_model)
        self.gru = nn.GRU(d_model, d_model, num_layers, batch_first=True, dropout=dropout)
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
    def forward(self, tok, gap):
        # tok: (B, seq_len), gap: (B, seq_len)
        tok_e = self.token_emb(tok.long())
        gap_e = self.gap_emb(gap.long())
        x = self.dropout(tok_e + gap_e)
        _, h = self.gru(x)   # h: (num_layers, B, d_model)
        return h[-1]          # last layer output (B, d_model)

class DualGRU(nn.Module):
    def __init__(self, tab_dim, token_vocab, gap_vocab, seq_len, d_model=64, tab_h=128):
        super().__init__()
        self.lead_enc = GRUEncoder(token_vocab, gap_vocab, seq_len, d_model)
        self.rep_enc = GRUEncoder(token_vocab, gap_vocab, seq_len, d_model)
        self.tab_net = nn.Sequential(
            nn.Linear(tab_dim, tab_h), nn.ReLU(),
            nn.Linear(tab_h, tab_h), nn.ReLU()
        )
        self.out_dim = tab_h + 2*d_model
    def forward(self, tab_x, lead_tok, lead_gap, rep_tok, rep_gap):
        z_tab = self.tab_net(tab_x)
        z_lead = self.lead_enc(lead_tok, lead_gap)
        z_rep = self.rep_enc(rep_tok, rep_gap)
        return torch.cat([z_tab, z_lead, z_rep], dim=1)

class QNet(nn.Module):
    def __init__(self, tab_dim, token_vocab, gap_vocab, seq_len, n_actions):
        super().__init__()
        self.enc = DualGRU(tab_dim, token_vocab, gap_vocab, seq_len)
        self.head = nn.Sequential(nn.Linear(self.enc.out_dim, 128), nn.ReLU(),
                                  nn.Linear(128, n_actions))
    def forward(self, tab_x, lead_tok, lead_gap, rep_tok, rep_gap):
        z = self.enc(tab_x, lead_tok, lead_gap, rep_tok, rep_gap)
        return self.head(z)

class VNet(nn.Module):
    def __init__(self, tab_dim, token_vocab, gap_vocab, seq_len):
        super().__init__()
        self.enc = DualGRU(tab_dim, token_vocab, gap_vocab, seq_len)
        self.head = nn.Sequential(nn.Linear(self.enc.out_dim, 128), nn.ReLU(),
                                  nn.Linear(128, 1))
    def forward(self, tab_x, lead_tok, lead_gap, rep_tok, rep_gap):
        z = self.enc(tab_x, lead_tok, lead_gap, rep_tok, rep_gap)
        return self.head(z).squeeze(1)

def expectile_loss(diff, tau=0.7):
    w = torch.where(diff > 0, tau, 1 - tau)
    return (w * diff.pow(2)).mean()

# Double Q-learning: two Q networks and a target Q network
Q1 = QNet(tab_dim, token_vocab, gap_vocab, HIST_N, K).to(device)
Q2 = QNet(tab_dim, token_vocab, gap_vocab, HIST_N, K).to(device)
Q_target = QNet(tab_dim, token_vocab, gap_vocab, HIST_N, K).to(device)
Q_target.load_state_dict(Q1.state_dict())
Vnet = VNet(tab_dim, token_vocab, gap_vocab, HIST_N).to(device)

optQ1 = optim.Adam(Q1.parameters(), lr=1e-4, weight_decay=1e-5)
optQ2 = optim.Adam(Q2.parameters(), lr=1e-4, weight_decay=1e-5)
optV = optim.Adam(Vnet.parameters(), lr=1e-4, weight_decay=1e-5)

schedulerQ1 = optim.lr_scheduler.ReduceLROnPlateau(optQ1, patience=5, factor=0.5)
schedulerQ2 = optim.lr_scheduler.ReduceLROnPlateau(optQ2, patience=5, factor=0.5)
schedulerV = optim.lr_scheduler.ReduceLROnPlateau(optV, patience=5, factor=0.5)

polyak = 0.995

In [34]:
# Tiny example with shapes

# Suppose:

# B = 32
# seq_len = 10
# tab_dim = 76
# d_model = 64
# tab_h = 128

# Then:

# Inputs
# tab_x: (32, 76)
# lead_tok: (32, 10)
# lead_gap: (32, 10)
# rep_tok: (32, 10)
# rep_gap: (32, 10)
# After encoding
# z_tab: (32, 128)
# z_lead: (32, 64)
# z_rep: (32, 64)
# Concatenated
# z: (32, 256)
# If passed to QNet.head
# output: (32, K)
# If passed to VNet.head
# output: (32,)

## 8) IQL training with Double Q, target network, and capacity‑aware policy evaluation (offline)

We train the IQL agent, then later apply a capacity‑constrained greedy policy.

In [51]:
S_lead_gap[0:10]

array([[-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
       [ 1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
       [ 1.,  2., -1., -1., -1., -1., -1., -1., -1., -1.],
       [ 1.,  2.,  3., -1., -1., -1., -1., -1., -1., -1.],
       [ 1.,  2.,  3.,  4., -1., -1., -1., -1., -1., -1.],
       [ 1.,  2.,  3.,  4.,  5., -1., -1., -1., -1., -1.],
       [ 1.,  2.,  3.,  4.,  5.,  6., -1., -1., -1., -1.],
       [ 2.,  3.,  4.,  5.,  6.,  7., -1., -1., -1., -1.],
       [ 1.,  3.,  4.,  5.,  6.,  7.,  8., -1., -1., -1.],
       [ 1.,  2.,  4.,  5.,  6.,  7.,  8.,  9., -1., -1.]], dtype=float32)

In [52]:
lead_gap_disc[0:10]

array([[6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [1, 6, 6, 6, 6, 6, 6, 6, 6, 6],
       [1, 2, 6, 6, 6, 6, 6, 6, 6, 6],
       [1, 2, 3, 6, 6, 6, 6, 6, 6, 6],
       [1, 2, 3, 4, 6, 6, 6, 6, 6, 6],
       [1, 2, 3, 4, 5, 6, 6, 6, 6, 6],
       [1, 2, 3, 4, 5, 5, 6, 6, 6, 6],
       [2, 3, 4, 5, 5, 5, 6, 6, 6, 6],
       [1, 3, 4, 5, 5, 5, 5, 6, 6, 6],
       [1, 2, 4, 5, 5, 5, 5, 5, 6, 6]])

In [53]:
done

array([0., 0., 0., ..., 0., 0., 1.], shape=(21000,), dtype=float32)

In [35]:
# Pre‑discretize gaps

def discretize_gap(gap, max_bin=5):
    if np.isnan(gap) or gap < 0:
        return 6
    return min(int(gap), max_bin)

def apply_discretize(gap_array):
    res = np.zeros_like(gap_array, dtype=np.int64)
    for i in range(gap_array.shape[0]):
        for j in range(gap_array.shape[1]):
            res[i,j] = discretize_gap(gap_array[i,j])
    return res

lead_gap_disc = apply_discretize(S_lead_gap)
rep_gap_disc = apply_discretize(S_rep_gap)

lead_gap_next_disc = apply_discretize(S_next_lead_gap)
rep_gap_next_disc = apply_discretize(S_next_rep_gap)

S_tab_t = torch.from_numpy(S_tab).to(device)

S_lead_tok_t = torch.from_numpy(S_lead_tok).to(device)
S_lead_gap_t = torch.from_numpy(lead_gap_disc).to(device)

S_rep_tok_t = torch.from_numpy(S_rep_tok).to(device)
S_rep_gap_t = torch.from_numpy(rep_gap_disc).to(device)

Sn_tab_t = torch.from_numpy(S_next_tab).to(device)

Sn_lead_tok_t = torch.from_numpy(S_next_lead_tok).to(device)
Sn_lead_gap_t = torch.from_numpy(lead_gap_next_disc).to(device)

Sn_rep_tok_t = torch.from_numpy(S_next_rep_tok).to(device)
Sn_rep_gap_t = torch.from_numpy(rep_gap_next_disc).to(device)

# A = snap_sorted["chosen_action"].map(a_to_idx).astype(int).values
# R = snap_sorted["reward"].astype(np.float32).values

A_t = torch.from_numpy(A.astype(np.int64)).to(device)
R_t = torch.from_numpy(R.astype(np.float32)).to(device)
D_t = torch.from_numpy(done.astype(np.float32)).to(device)

In [39]:
S_lead_gap[10]

array([ 1.,  2.,  3.,  5.,  6.,  7.,  8.,  9., 10., -1.], dtype=float32)

In [40]:
lead_gap_disc[10]

array([1, 2, 3, 5, 5, 5, 5, 5, 5, 6])

In [ ]:
# target = reward + gamma * (1 - done) * next_value

In [22]:


idx_all = np.arange(len(S_tab))
rl_tr, rl_va = train_test_split(idx_all, test_size=0.2, random_state=SEED)


# batch size = 512
# max epochs = 50
# gamma = discount factor
# tau = expectile parameter for value learning

bs = 512
epochs = 50
gamma = cfg.gamma
tau = 0.7

best_val_loss = float('inf')
patience = 10
wait = 0

# rng = np.random.default_rng(SEED)

for ep in range(1, epochs+1):
    rng.shuffle(rl_tr)
    q1_losses, q2_losses, v_losses = [], [], []
    Q1.train(); Q2.train(); Vnet.train()
    
    for start in range(0, len(rl_tr), bs):
        b = rl_tr[start:start+bs]
        
        s_tab_b = S_tab_t[b]; 
        
        sL_tok_b = S_lead_tok_t[b]; 
        sL_gap_b = S_lead_gap_t[b]
        
        sR_tok_b = S_rep_tok_t[b];
        sR_gap_b = S_rep_gap_t[b]
        
        sn_tab_b = Sn_tab_t[b];
        
        snL_tok_b = Sn_lead_tok_t[b];
        snL_gap_b = Sn_lead_gap_t[b]
        
        snR_tok_b = Sn_rep_tok_t[b];
        snR_gap_b = Sn_rep_gap_t[b]
        
        a_b = A_t[b]; 
        r_b = R_t[b]; 
        d_b = D_t[b]

        with torch.no_grad():
            next_v = Vnet(sn_tab_b, snL_tok_b, snL_gap_b, snR_tok_b, snR_gap_b)
            # y=r+γ(1−d)V(s′)
            # Where:
            # r = reward now
            # gamma = discount factor
            # d = done flag
            # V(s') = value of next state
            
            y = r_b + gamma*(1.0 - d_b)*next_v

        q1_sa = Q1(s_tab_b, sL_tok_b, sL_gap_b, sR_tok_b, sR_gap_b).gather(1, a_b.view(-1,1)).squeeze(1)
        q2_sa = Q2(s_tab_b, sL_tok_b, sL_gap_b, sR_tok_b, sR_gap_b).gather(1, a_b.view(-1,1)).squeeze(1)

        # So the Q-networks are being trained to predict the Bellman target.
        lossQ1 = nn.functional.mse_loss(q1_sa, y)
        lossQ2 = nn.functional.mse_loss(q2_sa, y)

        optQ1.zero_grad(); 
        lossQ1.backward(); 
        nn.utils.clip_grad_norm_(Q1.parameters(), 1.0); 
        optQ1.step()
        
        optQ2.zero_grad(); 
        lossQ2.backward(); 
        nn.utils.clip_grad_norm_(Q2.parameters(), 1.0); 
        optQ2.step()
        
        q1_losses.append(float(lossQ1.item()))
        q2_losses.append(float(lossQ2.item()))

        # V update using the minimum Q (conservative)
        with torch.no_grad():
            q1_det = Q1(s_tab_b, sL_tok_b, sL_gap_b, sR_tok_b, sR_gap_b).gather(1, a_b.view(-1,1)).squeeze(1)
            q2_det = Q2(s_tab_b, sL_tok_b, sL_gap_b, sR_tok_b, sR_gap_b).gather(1, a_b.view(-1,1)).squeeze(1)
            q_det = torch.min(q1_det, q2_det)
            
        v = Vnet(s_tab_b, sL_tok_b, sL_gap_b, sR_tok_b, sR_gap_b)
        lossV = expectile_loss(q_det - v, tau=tau)
        optV.zero_grad(); lossV.backward(); nn.utils.clip_grad_norm_(Vnet.parameters(), 1.0); optV.step()
        v_losses.append(float(lossV.item()))

    # Polyak update of target Q
    for param, target_param in zip(Q1.parameters(), Q_target.parameters()):
        target_param.data.copy_(polyak * target_param.data + (1 - polyak) * param.data)

    # Validation
    Q1.eval(); Q2.eval(); Vnet.eval()
    val_q_losses = []
    with torch.no_grad():
        for start in range(0, len(rl_va), bs):
            b = rl_va[start:start+bs]
            s_tab_b = S_tab_t[b]; 
            
            sL_tok_b = S_lead_tok_t[b]; 
            sL_gap_b = S_lead_gap_t[b]
            
            sR_tok_b = S_rep_tok_t[b]; 
            
            sR_gap_b = S_rep_gap_t[b]
            sn_tab_b = Sn_tab_t[b]; 
            
            snL_tok_b = Sn_lead_tok_t[b]; 
            snL_gap_b = Sn_lead_gap_t[b]
            
            snR_tok_b = Sn_rep_tok_t[b]; 
            snR_gap_b = Sn_rep_gap_t[b]
            
            a_b = A_t[b]; 
            r_b = R_t[b]; 
            d_b = D_t[b]
            
            next_v = Vnet(sn_tab_b, snL_tok_b, snL_gap_b, snR_tok_b, snR_gap_b)
            
            y = r_b + gamma*(1.0 - d_b)*next_v
            q1_sa = Q1(s_tab_b, sL_tok_b, sL_gap_b, sR_tok_b, sR_gap_b).gather(1, a_b.view(-1,1)).squeeze(1)
            val_q_losses.append(float(nn.functional.mse_loss(q1_sa, y).item()))
            
    avg_val_loss = np.mean(val_q_losses)
    schedulerQ1.step(avg_val_loss); schedulerQ2.step(avg_val_loss); schedulerV.step(avg_val_loss)

    print(f"Epoch {ep}/{epochs}: Q1_loss={np.mean(q1_losses):.4f}, Q2_loss={np.mean(q2_losses):.4f}, V_loss={np.mean(v_losses):.4f}, Val_Q_loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        wait = 0
        torch.save(Q1.state_dict(), "best_Q1.pt")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {ep}")
            break

# Load best Q1
Q1.load_state_dict(torch.load("best_Q1.pt"))
Q1.eval()

Epoch 1/50: Q1_loss=0.3048, Q2_loss=0.2985, V_loss=0.0094, Val_Q_loss=0.3373
Epoch 2/50: Q1_loss=0.3854, Q2_loss=0.3717, V_loss=0.0319, Val_Q_loss=0.4610
Epoch 3/50: Q1_loss=0.5275, Q2_loss=0.4084, V_loss=0.0471, Val_Q_loss=0.6230
Epoch 4/50: Q1_loss=0.6620, Q2_loss=0.4998, V_loss=0.0372, Val_Q_loss=0.6329
Epoch 5/50: Q1_loss=0.7572, Q2_loss=0.6973, V_loss=0.0213, Val_Q_loss=0.6893
Epoch 6/50: Q1_loss=0.8856, Q2_loss=0.8678, V_loss=0.0210, Val_Q_loss=0.7390
Epoch 7/50: Q1_loss=0.7526, Q2_loss=0.7367, V_loss=0.0235, Val_Q_loss=0.5869
Epoch 8/50: Q1_loss=0.6575, Q2_loss=0.6454, V_loss=0.0321, Val_Q_loss=0.5778
Epoch 9/50: Q1_loss=0.6765, Q2_loss=0.6667, V_loss=0.0365, Val_Q_loss=0.5742
Epoch 10/50: Q1_loss=0.6414, Q2_loss=0.6319, V_loss=0.0397, Val_Q_loss=0.5466
Epoch 11/50: Q1_loss=0.5972, Q2_loss=0.5924, V_loss=0.0408, Val_Q_loss=0.5169
Early stopping at epoch 11


QNet(
  (enc): DualGRU(
    (lead_enc): GRUEncoder(
      (token_emb): Embedding(7, 64)
      (gap_emb): Embedding(8, 64)
      (gru): GRU(64, 64, num_layers=2, batch_first=True, dropout=0.1)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (rep_enc): GRUEncoder(
      (token_emb): Embedding(7, 64)
      (gap_emb): Embedding(8, 64)
      (gru): GRU(64, 64, num_layers=2, batch_first=True, dropout=0.1)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (tab_net): Sequential(
      (0): Linear(in_features=76, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): ReLU()
    )
  )
  (head): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=6, bias=True)
  )
)

## 9) Capacity‑aware target policy

We now produce a policy that respects each rep’s daily capacity. For each day and rep, we greedily assign actions to leads based on Q‑values, respecting the cost budget.

In [23]:
# Prepare test data
Xte = X_tab[te_idx]
LT_tok_te = lead_tok_all[te_idx]
LT_gap_te = apply_discretize(lead_gap_all[te_idx])
RT_tok_te = rep_tok_all[te_idx]
RT_gap_te = apply_discretize(rep_gap_all[te_idx])
rep_ids_te = snap.loc[te_idx, "rep_id"].values
daily_caps = snap.loc[te_idx, "daily_capacity"].values

with torch.no_grad():
    Xte_t = torch.from_numpy(Xte).to(device)
    LT_tok_t = torch.from_numpy(LT_tok_te).to(device)
    LT_gap_t = torch.from_numpy(LT_gap_te).to(device)
    RT_tok_t = torch.from_numpy(RT_tok_te).to(device)
    RT_gap_t = torch.from_numpy(RT_gap_te).to(device)
    Q_vals = Q1(Xte_t, LT_tok_t, LT_gap_t, RT_tok_t, RT_gap_t).cpu().numpy().astype(np.float32)

# Mask invalid actions using candidate_actions
cand_list = [parse_cand(snap.loc[i, "candidate_actions"]) for i in te_idx]
mask = np.full((len(te_idx), K), False)
for i, cands in enumerate(cand_list):
    mask[i, cands] = True
Q_masked = Q_vals.copy()
Q_masked[~mask] = -1e9

# Capacity‑constrained assignment: greedy per (rep, day)
# First, group transitions by (rep_id, date) – we need date from snap
dates_te = snap.loc[te_idx, "recommendation_date"].values
rep_dates = [(rep_ids_te[i], dates_te[i]) for i in range(len(te_idx))]

# We'll create a mapping from index to group
from collections import defaultdict
groups = defaultdict(list)
for i, rd in enumerate(rep_dates):
    groups[rd].append(i)

a_star_capacity = np.zeros(len(te_idx), dtype=int)
for (rep, date), indices in groups.items():
    cap = daily_caps[indices[0]]
    # Sort indices by the maximum Q-value for that state (descending)
    sorted_idx = sorted(indices, key=lambda x: np.max(Q_masked[x]), reverse=True)
    remaining = cap
    for idx in sorted_idx:
        best_action = np.argmax(Q_masked[idx])
        cost = ACTION_COST[idx_to_a[best_action]]
        if cost <= remaining:
            a_star_capacity[idx] = best_action
            remaining -= cost
        else:
            a_star_capacity[idx] = a_to_idx["wait"]

print("Capacity‑aware RL policy action distribution (test):")
print(pd.Series(a_star_capacity).map(idx_to_a).value_counts(normalize=True).round(3))

Capacity‑aware RL policy action distribution (test):
nurture    1.0
Name: proportion, dtype: float64


In [24]:
# OPE using the same Q1 network as baseline
ips_rl, dr_rl = ope_ips_dr(a_logged_te, r_logged_te, pb_te, a_star_capacity, Q_vals, clip=10.0)
print(f"OPE (RL capacity‑aware) IPS: {ips_rl:.4f}, DR: {dr_rl:.4f}")
ci_rl = bootstrap_ci_ope(a_logged_te, r_logged_te, pb_te, a_star_capacity, Q_vals, n_boot=300, clip=10.0)
ci_rl

OPE (RL capacity‑aware) IPS: 0.3338, DR: 0.3473


{'ips_mean': 0.33418169326068503,
 'ips_lo': 0.31198956777317616,
 'ips_hi': 0.3566964513948884,
 'dr_mean': 0.3467619171676418,
 'dr_lo': 0.32572625640786285,
 'dr_hi': 0.36987586015773316}

## 10) Sanity check: logging policy

In [25]:
a_logged_star = a_logged_te
ips_log, dr_log = ope_ips_dr(a_logged_te, r_logged_te, pb_te, a_logged_star, Q_vals, clip=10.0)
empirical_mean = r_logged_te.mean()
print(f"Empirical mean reward: {empirical_mean:.4f}")
print(f"IPS for logging policy: {ips_log:.4f} (should equal empirical mean)")
print(f"DR for logging policy: {dr_log:.4f}")

Empirical mean reward: 0.3019
IPS for logging policy: 1.1435 (should equal empirical mean)
DR for logging policy: 0.2680


## 11) Ablation summary table

In [26]:
ablation = pd.DataFrame([
    {"policy": "Bandit (argmax Q̂)", **ci_bandit},
    {"policy": "Offline RL (IQL, QNet, capacity-aware)", **ci_rl},
])
ablation[["policy","ips_mean","ips_lo","ips_hi","dr_mean","dr_lo","dr_hi"]]

,policy,ips_mean,ips_lo,ips_hi,dr_mean,dr_lo,dr_hi
0,Bandit (argmax Q̂),0.334182,0.31199,0.356696,0.468143,0.446246,0.486835
1,"Offline RL (IQL, QNet, capacity-aware)",0.334182,0.31199,0.356696,0.346762,0.325726,0.369876


## 12) Hyperparameter tuning recommendations

For production use, consider tuning the following with a grid search:
- `tau` ∈ {0.7, 0.8, 0.9}
- `lr` ∈ {1e-4, 5e-5, 1e-5}
- `d_model` ∈ {32, 64, 128}
- `gamma` ∈ {0.95, 0.98, 0.99}
- `polyak` ∈ {0.99, 0.995, 0.999}

Use validation Q‑loss as the selection criterion.